<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    ML con docking: ChEMBL → Preparación → Vina → ProLIF → Modelo → Ranking
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 6 — Pipeline completo de virtual screening</em>
  </p>
</div>


---
## Contexto: ¿dónde estamos en el curso?

Este notebook es el cierre del **triángulo de datos → modelos → estructura**
que hemos construido a lo largo de las semanas:

```
Semana 2-3:  ChEMBL → colección y curación de datos  (NB-DATA-01/02)
Semana 3:    Features y espacio químico               (NB-DATA-03)
Semana 4:    Modelos QSAR — clasificación activo/inactivo (NB-ML-01)
Semana 6:    Re-docking y validación del protocolo    (NB-DOCK-01)
Semana 6:    Docking en batch + scoring compuesto     ← ESTE NOTEBOOK
```

En NB-DOCK-01 validamos que nuestro protocolo reproduce la pose cristalográfica (RMSD ≤ 2 Å).
Ahora lo aplicamos a **todas las moléculas activas de ChEMBL** para nuestro target.

### ¿Por qué el score de Vina solo no es suficiente?

El score de Vina (kcal/mol) es una estimación de la energía de unión, pero dos moléculas
con el mismo score pueden tener **patrones de interacción completamente distintos**.
Una puede hacer puentes de hidrógeno con residuos clave del sitio catalítico;
la otra puede estar en una orientación inactiva.

Por eso construimos un **scoring compuesto** que combina:

$$\text{Score final} = \frac{\text{Score Vina normalizado} + \text{Similitud coseno ProLIF}}{2}$$

- **Score Vina normalizado:** energía de unión predicha (invertida y escalada a [0,1])
- **Similitud coseno ProLIF:** qué tan similares son las interacciones al ligando de referencia

Esto nos da un ranking que integra **energía** + **especificidad de interacciones**.

### ¿Qué produciremos?

| Salida | Descripción |
|--------|-------------|
| `docking/*.sdf` | Poses de todas las moléculas activas |
| `fps_activos.csv` | Fingerprints ProLIF de todas las poses |
| `ranking_final.csv` | Tabla con score Vina + similitud ProLIF + score compuesto |
| Top candidatos | Las mejores moléculas según el scoring compuesto |


---
## 1. Instalación y configuración

In [ ]:
# ── Instalar librerías ──────────────────────────────────────────────────────
!pip install chembl_webresource_client chembl-structure-pipeline rdkit \
             openbabel-wheel meeko tqdm vina MDAnalysis prolif \
             scikit-learn scipy --quiet

print("✅ Librerías instaladas")


In [ ]:
# ── Importaciones globales ──────────────────────────────────────────────────
import os, re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from math import log
from collections import defaultdict
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

# ChEMBL
from chembl_webresource_client.new_client import new_client
from chembl_structure_pipeline import standardize_mol

# Docking
from meeko import MoleculePreparation, PDBQTWriterLegacy
from openbabel import openbabel
from vina import Vina
import MDAnalysis as mda

# ProLIF y scoring
import prolif as plf
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import distance

# Habilitar widgets en Colab
from google.colab import output
output.enable_custom_widget_manager()

print("✅ Todo listo")


In [ ]:
# ── Parámetros del sistema — AJUSTA AQUÍ PARA TU PROYECTO ───────────────────
CHEMBL_TARGET_ID = 'CHEMBL203'    # ← ChEMBL ID de tu target (EGFR como ejemplo)
PDB_ID           = '7WJO'         # ← Código PDB (debe coincidir con NB-DOCK-01)
LIGAND_CODE      = 'BGI'          # ← Código del ligando de referencia (NB-DOCK-01)

# Carpetas de trabajo (deben existir desde NB-DOCK-01)
protein_directory = 'estructuras'
pdbqt_directory   = 'pdbqt'
molecules_directory = 'mols'
docks_directory     = 'docking'

# Crear carpetas nuevas
os.makedirs(molecules_directory, exist_ok=True)
os.makedirs(docks_directory, exist_ok=True)

print(f"Target:  {CHEMBL_TARGET_ID}")
print(f"Proteína: {PDB_ID}  |  Ligando referencia: {LIGAND_CODE}")
print()
print("Carpetas:")
for d in [protein_directory, pdbqt_directory, molecules_directory, docks_directory]:
    estado = '✅' if os.path.exists(d) else '⚠️  (se creará)'
    print(f"  {d}/  {estado}")


---
## 2. Descarga y curación de moléculas activas desde ChEMBL

Descargamos todas las moléculas activas reportadas para nuestro target.

### Criterios de selección (integrados en `chembl_mols`)

| Criterio | Valor | Justificación |
|----------|-------|--------------|
| Tipo de actividad | IC50, Ki, EC50, Kd | Medidas de potencia directa |
| Unidades | nM | Para poder comparar y calcular pActividad |
| pActividad mínima | ≥ 5 (IC50 ≤ 10 µM) | Filtrar débiles/inactivos |
| Peso molecular | 180–900 Da | Rango drug-like |
| Duplicados | Solo SMILES únicos | Evitar sesgo en el análisis |

> 💡 Estos criterios son los mismos que discutimos en NB-DATA-01.
> Si ya tienes el CSV de NB-DATA-02 curado, puedes cargarlo directamente
> en lugar de descargar de nuevo (ver Opción B al final de la sección).


In [ ]:
# ── Función de descarga desde ChEMBL ────────────────────────────────────────
def chembl_mols(chembl_id):
    """
    Descarga moléculas activas de un target ChEMBL.

    Criterios:
    - Actividades: IC50, Ki, EC50, Kd en nM
    - pActividad ≥ 5 (IC50 ≤ 10 µM)
    - Peso molecular: 180–900 Da
    - Solo SMILES únicos (mayor pActividad si hay duplicados)

    Parámetros
    ----------
    chembl_id : str — ChEMBL ID del target (ej. 'CHEMBL203')

    Retorna
    -------
    df          : DataFrame con las moléculas
    target_name : str — nombre del target
    organism    : str — organismo
    """
    activity_client = new_client.activity
    target_client   = new_client.target

    # Descargar actividades
    actividades = activity_client.filter(
        target_chembl_id=chembl_id
    ).filter(
        standard_units='nM'
    ).only(
        'canonical_smiles', 'molecule_chembl_id',
        'pchembl_value', 'standard_units',
        'standard_value', 'standard_type'
    )

    if not actividades:
        print(f"❌ No se encontraron actividades para {chembl_id}")
        return None, None, None

    # Info del target
    target_info = target_client.filter(
        target_chembl_id=chembl_id
    ).only('pref_name', 'organism')
    target_df   = pd.DataFrame(target_info)
    target_name = target_df['pref_name'].iloc[0]
    organism    = target_df['organism'].iloc[0]

    # Procesar
    df = pd.DataFrame(actividades)
    df = df[df['standard_type'].isin(['IC50', 'Ki', 'EC50', 'Kd'])]
    df = df.astype({'standard_value': float})
    df = df[df['standard_value'] > 0]

    # Calcular pActividad = -log10(valor en Molar)
    df['pValue'] = [-log(v / 1e9, 10) for v in df['standard_value']]

    # Deduplicar por SMILES — conservar el de mayor pActividad
    df = df.sort_values('pValue', ascending=False)
    df = df.drop_duplicates(subset=['canonical_smiles'], keep='first').reset_index(drop=True)

    # Filtrar por pActividad y peso molecular
    df = df[df['pValue'] >= 5]

    mw = []
    for smi in df['canonical_smiles']:
        try:
            mol = Chem.MolFromSmiles(smi)
            mw.append(Descriptors.MolWt(mol) if mol else 0)
        except:
            mw.append(0)
    df['mol_weight'] = mw
    df = df[(df['mol_weight'] >= 180) & (df['mol_weight'] <= 900)].reset_index(drop=True)

    return df, target_name, organism


In [ ]:
# ── Opción A: Descargar directamente desde ChEMBL ────────────────────────────
print(f"Descargando moléculas activas para {CHEMBL_TARGET_ID}...")
print("Esto puede tardar 1-2 minutos...")
print()

chembl_list, TARGET_NAME, TARGET_ORG = chembl_mols(CHEMBL_TARGET_ID)

print(f"✅ Target: {TARGET_NAME} ({TARGET_ORG})")
print(f"   Moléculas descargadas: {len(chembl_list)}")
print(f"   pActividad — media: {chembl_list['pValue'].mean():.2f}  "
      f"rango: [{chembl_list['pValue'].min():.1f}, {chembl_list['pValue'].max():.1f}]")
print()
chembl_list.head()


In [ ]:
# ── Opción B: Cargar el CSV curado de NB-DATA-02 (más rápido) ────────────────
# Si ya tienes el CSV de NB-DATA-02, descomenta y ajusta el nombre:

# chembl_list = pd.read_csv('dataset_<target>_curado.csv')
# chembl_list = chembl_list.rename(columns={
#     'std_smiles': 'canonical_smiles',
#     'pActividad': 'pValue'
# })
# TARGET_NAME = 'Tu target'
# print(f"✅ {len(chembl_list)} moléculas cargadas desde NB-DATA-02")

print("💡 Usando datos descargados directamente de ChEMBL (Opción A)")
print("   Para usar el CSV de NB-DATA-02, descomenta el bloque de arriba.")


---
## 3. Estandarización de SMILES

Aplicamos el mismo pipeline de curación estructural que en NB-DATA-02:
`read_smiles` → `select_largest_organic_component` → `chembl_standardizer`

Esto garantiza que los SMILES son válidos, sin sales y en formato canónico ChEMBL
antes de generar los conformeros 3D para el docking.


In [ ]:
# ── Funciones de curación (reutilizadas de NB-DATA-02) ───────────────────────

def read_smiles(smiles):
    """Verifica si el SMILES es válido con RDKit."""
    if pd.isna(smiles) or str(smiles).strip() == '':
        return None, "SMILES vacío"
    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None:
        return None, "SMILES inválido"
    return mol, None

def is_valid(mol):
    """Verifica que el fragmento sea una molécula orgánica drug-like."""
    organic_elements = {'H','B','C','N','O','F','Si','P','S','Cl','Br','I'}
    for atom in mol.GetAtoms():
        if atom.GetSymbol() not in organic_elements:
            return False
    if Descriptors.RingCount(mol) < 1:
        return False
    if Descriptors.NumHAcceptors(mol) < 1:
        return False
    return True

def select_largest_organic_component(mol):
    """Elimina sales y selecciona el fragmento orgánico principal."""
    common_smiles = {
        'Cc1ccc(S(=O)(=O)[O-])cc1', 'O=S(=O)([O-])c1ccccc1',
        'O=C(O)C(F)(F)F', 'O=C(O)C(=O)O',
    }
    fragments = list(Chem.GetMolFrags(mol, asMols=True))
    n_total = len(fragments)
    fragments = [f for f in fragments if is_valid(f)]
    fragments = [f for f in fragments if Chem.MolToSmiles(f) not in common_smiles]
    unique_smiles = list(set(Chem.MolToSmiles(f) for f in fragments))
    unique_frags  = [Chem.MolFromSmiles(s) for s in unique_smiles]
    if not unique_frags:
        return None, n_total, len(unique_frags), "Sin fragmento válido"
    if len(unique_frags) == 1:
        return unique_frags[0], n_total, 1, None
    largest = max(unique_frags, key=lambda x: x.GetNumHeavyAtoms())
    total_at = sum(f.GetNumHeavyAtoms() for f in unique_frags)
    if largest.GetNumHeavyAtoms() < 0.6 * total_at:
        return None, n_total, len(unique_frags), "Fragmento < 60% del total"
    return largest, n_total, len(unique_frags), None

def chembl_standardizer(mol):
    """Estandariza con el protocolo oficial ChEMBL Structure Pipeline."""
    try:
        mol_std = standardize_mol(mol)
        return Chem.MolToSmiles(mol_std), None
    except Exception as e:
        return None, str(e)[:60]

def process_smiles(smiles):
    """Pipeline completo: lectura → eliminación de sales → estandarización."""
    mol, err = read_smiles(smiles)
    if err:
        return None, {'paso': 1, 'error': err}
    frag, _, _, err = select_largest_organic_component(mol)
    if err:
        return None, {'paso': 2, 'error': err}
    smi_std, err = chembl_standardizer(frag)
    if err:
        return None, {'paso': 3, 'error': err}
    return smi_std, {'paso': None, 'error': None}

print("✅ Funciones de curación cargadas")


In [ ]:
# ── Aplicar la estandarización al dataset ───────────────────────────────────
print(f"Estandarizando {len(chembl_list)} SMILES...")

resultados = [process_smiles(smi) for smi in tqdm(chembl_list['canonical_smiles'])]
chembl_list['std_smiles'] = [r[0] for r in resultados]
chembl_list['cur_error']  = [r[1]['error'] for r in resultados]

# Eliminar moléculas que no pasaron la curación
n_antes = len(chembl_list)
chembl_list = chembl_list.dropna(subset=['std_smiles']).reset_index(drop=True)
n_despues = len(chembl_list)

print(f"\nRESULTADO DE LA ESTANDARIZACIÓN")
print(f"  Antes:   {n_antes}")
print(f"  Después: {n_despues}")
print(f"  Descartadas: {n_antes - n_despues} ({(n_antes-n_despues)/n_antes*100:.1f}%)")
print()
chembl_list[['molecule_chembl_id', 'std_smiles', 'pValue',
             'standard_type', 'standard_value']].head(5)


---
## 4. Generación de conformeros 3D

El docking necesita una representación 3D de cada molécula.
Generamos **múltiples conformeros** para cada SMILES y seleccionamos
el de menor energía — esta es la geometría de partida para Vina.

### Parámetros clave

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `numConfs` | 10 | Generar 10 conformeros candidatos |
| `pruneRmsThresh` | 1 Å | Eliminar conformeros muy similares entre sí |
| `ETKDGv3` | — | Algoritmo de generación (más moderno y preciso) |
| `UFFOptimize` | 200 iter | Minimización de energía con campo de fuerzas UFF |


In [ ]:
# ── Función de generación de conformeros ────────────────────────────────────
def get_conformers(smiles, name, path="."):
    """
    Genera conformeros 3D para una molécula y guarda el de menor energía en PDB.

    Pipeline:
    1. SMILES → molécula RDKit
    2. Añadir hidrógenos explícitos
    3. Generar 10 conformeros con ETKDGv3
    4. Optimizar con campo de fuerzas UFF
    5. Seleccionar el conformero de menor energía
    6. Guardar en PDB
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = Chem.AddHs(mol)

        # Generar conformeros con ETKDGv3
        # numConfs se pasa como argumento, NO como atributo de params
        params = AllChem.ETKDGv3()
        params.pruneRmsThresh = 1.0
        params.numThreads     = -1    # usar todos los núcleos
        params.randomSeed     = 42

        conformeros = AllChem.EmbedMultipleConfs(mol, numConfs=10, params=params)

        if len(conformeros) == 0:
            # Fallback: generación aleatoria si ETKDGv3 falla
            params_alt = AllChem.EmbedParameters()
            params_alt.useRandomCoords = True
            AllChem.EmbedMultipleConfs(mol, numConfs=10, params=params_alt)

        if mol.GetNumConformers() == 0:
            return None

        # Optimizar con campo de fuerzas UFF
        energias = AllChem.UFFOptimizeMoleculeConfs(mol, maxIters=200)

        # Seleccionar el conformero de menor energía
        energias_vals = [e[1] for e in energias if e[0] == 0]
        idx_mejor = energias_vals.index(min(energias_vals)) if energias_vals else 0

        # RemoveAllHs devuelve una nueva molécula — asignar el resultado
        mol_sin_h = Chem.RemoveAllHs(mol)
        writer = Chem.PDBWriter(f'{path}/{name}.pdb')
        writer.write(mol_sin_h, confId=idx_mejor)
        writer.close()

        return f'{path}/{name}.pdb'

    except Exception:
        return None

print("✅ Función get_conformers lista")

In [ ]:
# ── Generar conformeros para todas las moléculas ─────────────────────────────
print(f"Generando conformeros para {len(chembl_list)} moléculas...")
print("(puede tardar varios minutos según el tamaño del dataset)")
print()

errores_conf = 0
for row in tqdm(chembl_list.iloc, total=len(chembl_list)):
    resultado = get_conformers(row.std_smiles, row.molecule_chembl_id, molecules_directory)
    if resultado is None:
        errores_conf += 1

print(f"\n✅ Conformeros generados")
print(f"   Exitosos:  {len(chembl_list) - errores_conf}")
print(f"   Fallidos:  {errores_conf}")

# Verificar cuántos PDB se crearon realmente
import glob
n_pdbs = len(glob.glob(f'{molecules_directory}/*.pdb'))
print(f"   Archivos PDB en disco: {n_pdbs}")


---
## 5. Preparación de ligandos en formato PDBQT con Meeko

AutoDock Vina requiere el formato PDBQT — una extensión del PDB que incluye
**cargas parciales** y **tipos de átomo** del campo de fuerzas AutoDock 4.

**Meeko** es la herramienta oficial de preparación de ligandos para Vina:
- Asigna cargas Gasteiger a cada átomo
- Identifica los **enlaces rotables** (torsiones del ligando durante el docking)
- Mantiene solo los **hidrógenos polares** (los no polares están implícitos en Vina)
- Genera el string PDBQT que Vina lee directamente en memoria (sin escribir a disco)


In [ ]:
# ── Función de preparación de ligandos con Meeko ────────────────────────────
def prepare_ligands(name):
    """
    Convierte un PDB de conformero mínimo a formato PDBQT usando Meeko.

    Proceso:
    1. Leer el PDB con RDKit
    2. Añadir hidrógenos con coordenadas 3D
    3. Asignar cargas Gasteiger
    4. Identificar torsiones del ligando
    5. Serializar a string PDBQT (sin escribir a disco)

    Parámetros
    ----------
    name : str — nombre del archivo (sin extensión), debe estar en molecules_directory

    Retorna
    -------
    str o None — contenido PDBQT como string
    """
    try:
        ruta_pdb = f'{molecules_directory}/{name}.pdb'
        if not os.path.exists(ruta_pdb):
            return None

        mol = Chem.MolFromPDBFile(ruta_pdb, removeHs=False, sanitize=True)
        if mol is None:
            return None

        # Añadir hidrógenos con coordenadas 3D explícitas
        mol = AllChem.AddHs(mol, addCoords=True)

        # Configurar Meeko con cargas Gasteiger
        mk_prep = MoleculePreparation(charge_model="gasteiger")
        mk_mol  = mk_prep.prepare(mol)

        # Generar el string PDBQT
        pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(mk_mol[0])

        if not is_ok:
            return None
        return pdbqt_string

    except Exception:
        return None

print("✅ Función prepare_ligands lista")


In [ ]:
# ── Preparar todas las moléculas ─────────────────────────────────────────────
print(f"Preparando {len(chembl_list)} ligandos en formato PDBQT...")

chembl_list['pdbqt'] = [
    prepare_ligands(row.molecule_chembl_id)
    for row in tqdm(chembl_list.iloc, total=len(chembl_list))
]

# Filtrar las que fallaron
n_antes = len(chembl_list)
chembl_list = chembl_list[chembl_list['pdbqt'].notna()].reset_index(drop=True)
n_despues = len(chembl_list)

print(f"\n✅ Ligandos preparados: {n_despues}")
print(f"   Fallidos: {n_antes - n_despues}")

# Guardar checkpoint
chembl_list.to_csv(f'{molecules_directory}/molecules_prepared.csv', index=False)
print(f"   Checkpoint guardado: {molecules_directory}/molecules_prepared.csv")


---
## 6. Docking en batch con AutoDock Vina

Ahora ejecutamos el docking de **todas las moléculas** contra el mismo sitio de unión
que validamos en NB-DOCK-01. Usamos los mismos parámetros de la caja de docking.

### Estrategia de eficiencia

En lugar de crear un objeto Vina por cada molécula (muy lento), cargamos el receptor
**una sola vez** y solo cambiamos el ligando en cada iteración. Esto reduce el tiempo
de preparación de ~5 segundos a <0.1 segundos por molécula.

> ⏱️ Tiempo estimado: ~30 segundos por molécula con exhaustiveness=16.
> Para datasets de >500 moléculas, considera reducir a exhaustiveness=8.


In [ ]:
import json

# ── Cargar parámetros de acoplamiento desde NB-DOCK-01 ───────────────────────
params_path = f"{protein_directory}/docking_params.json"

if not os.path.exists(params_path):
    raise FileNotFoundError(
        f"No se encontró {params_path}.\n"
        "Ejecuta primero el notebook NB-DOCK-01 (13 - ReDocking Validacion) "
        "hasta la celda de guardado de parámetros."
    )

with open(params_path) as f:
    dp = json.load(f)

pocket_center = np.array(dp["pocket_center"])
ligand_box    = np.array(dp["ligand_box"])

print("CAJA DE DOCKING (cargada desde NB-DOCK-01)")
print("=" * 40)
print(f"  PDB:       {dp['PDB_ID']}")
print(f"  Ligando:   {dp['LIGAND_CODE']}")
print(f"  Margen:    {dp['margen_angstrom']} Å")
print(f"  Centro: [{pocket_center[0]:.2f}, {pocket_center[1]:.2f}, {pocket_center[2]:.2f}] Å")
print(f"  Caja:   [{ligand_box[0]:.2f}, {ligand_box[1]:.2f}, {ligand_box[2]:.2f}] Å")
print()
print("✅ Mismos parámetros que NB-DOCK-01 — protocolo validado")

In [ ]:
# ── Función de conversión PDBQT → SDF ───────────────────────────────────────
def pdbqt_to_sdf(pdbqt_string, smiles, output_sdf_path):
    """
    Convierte las poses de Vina (PDBQT string) a SDF con órdenes de enlace correctos.

    Asigna los órdenes de enlace desde el SMILES de referencia para que
    las herramientas downstream (ProLIF, RDKit) puedan procesar correctamente.
    También guarda el Vina score como propiedad del SDF.
    """
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("pdbqt", "pdb")

    mols, scores = [], []

    # Extraer scores de los comentarios de Vina
    for linea in pdbqt_string.split('\n'):
        if 'VINA RESULT' in linea:
            try:
                scores.append(float(linea.split()[3]))
            except (IndexError, ValueError):
                pass

    # Parsear modelos del PDBQT
    mol = openbabel.OBMol()
    obConversion.ReadString(mol, pdbqt_string)
    mols.append(openbabel.OBMol(mol))
    while obConversion.Read(mol):
        mols.append(openbabel.OBMol(mol))

    if not mols:
        return False

    smiles_mol = Chem.MolFromSmiles(smiles)
    escritor   = Chem.SDWriter(output_sdf_path)

    for i, ob_mol in enumerate(mols):
        sdf_temp  = obConversion.WriteString(ob_mol)
        rdkit_mol = Chem.MolFromMolBlock(sdf_temp, removeHs=False, sanitize=False)
        if rdkit_mol is None:
            continue
        try:
            rdkit_mol_bo = AllChem.AssignBondOrdersFromTemplate(smiles_mol, rdkit_mol)
        except Exception:
            rdkit_mol_bo = rdkit_mol

        score = scores[i] if i < len(scores) else 0.0
        rdkit_mol_bo.SetDoubleProp('vina_score', score)
        rdkit_mol_bo.SetIntProp('pose_id', i + 1)
        escritor.write(rdkit_mol_bo)

    escritor.close()
    return True

print("✅ Función pdbqt_to_sdf lista")


In [ ]:
# ── Función de docking individual ───────────────────────────────────────────
def vina_docking(protein_id, pdbqt_string, smiles, path, name,
                 exhaustiveness=16, n_poses=5):
    """
    Realiza el docking de una molécula y guarda las poses en SDF.

    Parámetros
    ----------
    protein_id    : str — ID del receptor (archivo {pdbqt_directory}/{protein_id}.pdbqt)
    pdbqt_string  : str — ligando en formato PDBQT
    smiles        : str — SMILES del ligando (para asignar órdenes de enlace)
    path          : str — directorio de salida
    name          : str — nombre del archivo de salida
    exhaustiveness: int — exhaustividad de búsqueda (16=estándar, 8=rápido)
    n_poses       : int — número de poses a guardar

    Retorna
    -------
    np.array de scores (kcal/mol) o None si falla
    """
    try:
        v = Vina(sf_name='vina', verbosity=0)
        v.set_receptor(f"{pdbqt_directory}/{protein_id}.pdbqt")
        v.set_ligand_from_string(pdbqt_string)
        v.compute_vina_maps(
            center=pocket_center.tolist(),
            box_size=ligand_box.tolist()
        )
        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)

        pdbqt_poses = v.poses(n_poses=n_poses)
        sdf_path    = f"{path}/{name}.sdf"
        pdbqt_to_sdf(pdbqt_poses, smiles, sdf_path)

        return v.energies()[:, 0]   # columna 0 = score total

    except Exception as e:
        return None

print("✅ Función vina_docking lista")


In [ ]:
# ── Ejecutar el docking en batch ─────────────────────────────────────────────
# Verificar si ya hay resultados parciales (permite reanudar si se interrumpe)
ya_procesados = set(
    f.split('/')[-1].replace('.sdf','')
    for f in glob.glob(f'{docks_directory}/*.sdf')
) if __import__('glob').glob(f'{docks_directory}/*.sdf') else set()

pendientes = chembl_list[~chembl_list['molecule_chembl_id'].isin(ya_procesados)]

print(f"Moléculas totales:     {len(chembl_list)}")
print(f"Ya procesadas:         {len(ya_procesados)}")
print(f"Pendientes de docking: {len(pendientes)}")
print()

if len(pendientes) == 0:
    print("✅ Todos los dockings ya están calculados")
else:
    print(f"Iniciando docking batch ({len(pendientes)} moléculas)...")
    print("⏱️  Tiempo estimado: ~30 seg/molécula con exhaustiveness=16")
    print()

    scores_dict = {}
    for row in tqdm(pendientes.iloc, total=len(pendientes)):
        scores = vina_docking(
            protein_id=PDB_ID,
            pdbqt_string=row.pdbqt,
            smiles=row.std_smiles,
            path=docks_directory,
            name=row.molecule_chembl_id,
            exhaustiveness=16,
            n_poses=5
        )
        if scores is not None:
            scores_dict[row.molecule_chembl_id] = scores[0]  # mejor score

    print(f"\n✅ Docking completado")
    print(f"   SDFs generados: {len(list(__import__('glob').glob(f'{docks_directory}/*.sdf')))}")


In [ ]:
# ── Extraer el mejor score de Vina para cada molécula ───────────────────────
import glob

def extraer_mejor_score(sdf_path):
    """Extrae el score de la mejor pose (pose 1) desde un SDF de Vina."""
    try:
        suppl = Chem.SDMolSupplier(sdf_path, removeHs=False)
        mol   = suppl[0]  # pose 1 = mejor score
        if mol and mol.HasProp('vina_score'):
            return float(mol.GetProp('vina_score'))
    except Exception:
        pass
    return None

# Construir tabla de resultados
resultados_scores = []
for _, row in chembl_list.iterrows():
    sdf_path = f"{docks_directory}/{row['molecule_chembl_id']}.sdf"
    score = extraer_mejor_score(sdf_path) if os.path.exists(sdf_path) else None
    resultados_scores.append({
        'molecule_chembl_id': row['molecule_chembl_id'],
        'std_smiles':         row['std_smiles'],
        'pActividad':         row['pValue'],
        'standard_type':      row['standard_type'],
        'vina_score':         score
    })

df_resultados = pd.DataFrame(resultados_scores)
df_resultados = df_resultados.dropna(subset=['vina_score']).reset_index(drop=True)

print(f"RESULTADOS DE DOCKING — {TARGET_NAME}")
print("=" * 50)
print(f"  Moléculas con docking exitoso: {len(df_resultados)}")
print(f"  Score Vina — media:  {df_resultados['vina_score'].mean():.2f} kcal/mol")
print(f"  Score Vina — mejor:  {df_resultados['vina_score'].min():.2f} kcal/mol")
print(f"  Score Vina — peor:   {df_resultados['vina_score'].max():.2f} kcal/mol")
print()
df_resultados.sort_values('vina_score').head(10)


---
## 7. Fingerprints ProLIF para todas las poses

Calculamos los fingerprints de interacción proteína-ligando para:
1. El **ligando de referencia** (cristal) — será nuestro vector de comparación
2. Todas las **poses del docking** — las compararemos contra el cristal

El resultado es una matriz donde cada fila es una molécula y cada columna
es un par (residuo, tipo de interacción).


In [ ]:
# ── Cargar la proteína para ProLIF ───────────────────────────────────────────
rdkit_prot = Chem.MolFromPDBFile(
    f'{protein_directory}/{PDB_ID}_prep.pdb',
    removeHs=False, sanitize=False
)
if rdkit_prot is None:
    rdkit_prot = Chem.MolFromPDBFile(
        f'{protein_directory}/{PDB_ID}_a.pdb',
        removeHs=False, sanitize=False
    )

protein_mol = plf.Molecule(rdkit_prot)
print(f"✅ Proteína cargada para ProLIF: {PDB_ID}")


In [ ]:
# ── Función para calcular el fingerprint ProLIF de un SDF ───────────────────
def get_prolif(sdf_path, name):
    """
    Calcula el fingerprint de interacciones ProLIF para un conjunto de poses.

    Parámetros
    ----------
    sdf_path : str — ruta al archivo SDF con las poses
    name     : str — nombre de la molécula (para el índice del DataFrame)

    Retorna
    -------
    DataFrame con el fingerprint, o None si falla
    """
    try:
        poses = plf.sdf_supplier(sdf_path)
        if not poses:
            return None

        fp = plf.Fingerprint(vicinity_cutoff=8.0, count=True)
        fp.run_from_iterable(poses, protein_mol)

        df_fp = fp.to_dataframe()
        if df_fp.empty:
            return None

        # Tomar solo la mejor pose (primera)
        df_fp = df_fp.iloc[[0]]
        df_fp['mol'] = name
        return df_fp

    except Exception:
        return None

print("✅ Función get_prolif lista")


In [ ]:
# ── Calcular fingerprint del ligando de referencia (cristal) ─────────────────
sdf_cristal = f'{protein_directory}/{LIGAND_CODE}_org.sdf'
fp_cristal = get_prolif(sdf_cristal, LIGAND_CODE)

if fp_cristal is not None:
    print(f"✅ Fingerprint del cristal ({LIGAND_CODE}):")
    print(f"   Columnas de interacción: {fp_cristal.shape[1]-1}")
else:
    print("⚠️  No se pudo calcular el fingerprint del cristal")
    print(f"   Verifica que existe: {sdf_cristal}")


In [ ]:
# ── Calcular fingerprints para todas las moléculas del docking ───────────────
print(f"Calculando fingerprints ProLIF para {len(df_resultados)} moléculas...")

fps_lista = []
for mol_id in tqdm(df_resultados['molecule_chembl_id']):
    sdf_path = f"{docks_directory}/{mol_id}.sdf"
    if os.path.exists(sdf_path):
        fp = get_prolif(sdf_path, mol_id)
        if fp is not None:
            fps_lista.append(fp)

# Añadir el ligando de referencia al inicio
if fp_cristal is not None:
    fps_lista = [fp_cristal] + fps_lista

# Consolidar en un DataFrame
fps_df = pd.concat(fps_lista, ignore_index=True)

# Procesar el DataFrame
fps_df = fps_df.drop_duplicates(subset=['mol'])
fps_df = fps_df.set_index('mol')
fps_df = fps_df.fillna(0.0)

# Mantener solo columnas donde al menos 2 moléculas tienen la interacción
fps_df = fps_df.loc[:, fps_df.sum(axis=0) > 1]
fps_df = fps_df.astype(int)

print(f"\n✅ Matriz de fingerprints ProLIF")
print(f"   Moléculas: {fps_df.shape[0]}")
print(f"   Interacciones únicas: {fps_df.shape[1]}")

# Guardar
fps_df.to_csv(f'{docks_directory}/fps_activos.csv')
print(f"   Guardado: {docks_directory}/fps_activos.csv")


---
## 8. Visualización: heatmap de interacciones por tipo

Visualizamos la matriz de fingerprints agrupando las columnas por tipo de interacción.
Esto nos permite identificar de un vistazo:
- ¿Qué residuos son contactados más frecuentemente?
- ¿Qué tipos de interacción dominan el sitio de unión?
- ¿Qué moléculas tienen patrones similares al cristal (primera fila)?


In [ ]:
# ── Agrupar columnas por tipo de interacción ─────────────────────────────────
grouped_columns = defaultdict(list)

for col in fps_df.columns:
    col_str = str(col)
    # Intentar extraer el tipo de interacción del nombre de la columna
    # Formato esperado: 'Residuo.X_TipoInteraccion' o similar
    try:
        if '.X_' in col_str:
            tipo = col_str.split('.X_')[1]
        elif isinstance(col, tuple) and len(col) >= 2:
            tipo = col[1]
        else:
            # Último recurso: tomar la última parte después del punto
            partes = col_str.split('.')
            tipo = partes[-1] if partes else 'Other'
        grouped_columns[tipo].append(col)
    except Exception:
        grouped_columns['Other'].append(col)

print("TIPOS DE INTERACCIÓN ENCONTRADOS")
print("=" * 45)
for tipo, cols in sorted(grouped_columns.items(), key=lambda x: -len(x[1])):
    print(f"  {tipo:<20}: {len(cols):>3} residuos")


In [ ]:
# ── Heatmap por tipo de interacción ─────────────────────────────────────────
# Colores por tipo — consistente con la paleta del curso
c_maps = {
    'HBDonor':     'Blues',
    'HBAcceptor':  'Oranges',
    'Hydrophobic': 'Greens',
    'VdWContact':  'Greys',
    'PiStacking':  'Reds',
    'PiCation':    'Purples',
    'Cationic':    'YlOrRd',
    'Anionic':     'YlGnBu',
}

interaction_types = [t for t in grouped_columns if grouped_columns[t]]
col_widths = [len(grouped_columns[t]) for t in interaction_types]

if not interaction_types:
    print("⚠️  No hay suficientes interacciones para el heatmap")
else:
    fig, axes = plt.subplots(
        1, len(interaction_types),
        figsize=(max(12, sum(col_widths) * 0.6), max(6, len(fps_df) * 0.25)),
        sharey=True,
        gridspec_kw={'width_ratios': col_widths}
    )
    if len(interaction_types) == 1:
        axes = [axes]

    fig.subplots_adjust(wspace=0.02)

    for ax, tipo in zip(axes, interaction_types):
        cols_tipo = grouped_columns[tipo]
        data_tipo = fps_df[cols_tipo]

        # Nombres de columnas más legibles
        col_labels = [str(c).split('.')[-1][:8] if '.' in str(c) else str(c)[:8]
                      for c in cols_tipo]

        cmap = c_maps.get(tipo, 'Blues')
        sns.heatmap(
            data_tipo,
            ax=ax,
            cmap=cmap,
            cbar=False,
            xticklabels=col_labels,
            yticklabels=(ax == axes[0]),
            linewidths=0.2,
            linecolor='white'
        )
        ax.set_title(tipo, fontsize=9, fontweight='bold', pad=4)
        ax.tick_params(axis='x', labelsize=7, rotation=90)
        ax.tick_params(axis='y', labelsize=7)

    # Resaltar el ligando de referencia
    if LIGAND_CODE in fps_df.index:
        ref_idx = list(fps_df.index).index(LIGAND_CODE)
        for ax in axes:
            ax.axhline(ref_idx, color='yellow', linewidth=1.5, alpha=0.8)
            ax.axhline(ref_idx + 1, color='yellow', linewidth=1.5, alpha=0.8)

    fig.suptitle(
        f'Fingerprints de Interacción ProLIF — {TARGET_NAME}\n'
        f'(fila amarilla = ligando de referencia {LIGAND_CODE})',
        fontsize=11, fontweight='bold', y=1.01
    )
    plt.savefig(f'{docks_directory}/heatmap_interacciones.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    print(f"✅ Heatmap guardado: {docks_directory}/heatmap_interacciones.png")


---
## 9. Scoring compuesto: Vina + similitud ProLIF

### ¿Por qué combinar dos scores?

| Score solo | Limitación |
|-----------|-----------|
| Vina (energía) | Dos moléculas con el mismo score pueden unirse de forma completamente diferente |
| ProLIF (similitud) | Una molécula puede tener interacciones similares pero en una posición desfavorable energéticamente |
| **Combinado** | Selecciona moléculas que son **energéticamente favorables** Y **farmacológicamente específicas** |

### Fórmula del scoring compuesto

$$\text{Score final} = \frac{S_{\text{Vina}}^{\text{norm}} + S_{\text{ProLIF}}^{\text{coseno}}}{2}$$

Donde:
- $S_{\text{Vina}}^{\text{norm}}$: score de Vina invertido y normalizado a [0,1] con MinMaxScaler
  (el más negativo → 1.0; el menos negativo → 0.0)
- $S_{\text{ProLIF}}^{\text{coseno}}$: similitud coseno entre el fingerprint ProLIF de la molécula
  y el del ligando de referencia (cristal)


In [ ]:
# ── Paso 1: Calcular la similitud coseno de ProLIF vs cristal ─────────────────
if LIGAND_CODE not in fps_df.index:
    print(f"⚠️  '{LIGAND_CODE}' no está en el índice de fps_df")
    print(f"   Índices disponibles: {fps_df.index[:5].tolist()}")
    referencia_fp = fps_df.iloc[0]
    print(f"   Usando '{fps_df.index[0]}' como referencia")
else:
    referencia_fp = fps_df.loc[LIGAND_CODE]

# Escalar los fingerprints a [0,1]
scaler = MinMaxScaler()
fps_scaled_arr = scaler.fit_transform(fps_df.values.astype(float))
fps_scaled = pd.DataFrame(fps_scaled_arr,
                           index=fps_df.index,
                           columns=fps_df.columns)

ref_vector = fps_scaled.loc[referencia_fp.name] if hasattr(referencia_fp, 'name') else fps_scaled.iloc[0]

def similitud_coseno(row):
    """Calcula 1 - distancia coseno (similitud) entre una fila y el vector de referencia."""
    return 1 - distance.cosine(ref_vector.values, row.values)

fps_scaled['cos_sim'] = fps_scaled.apply(similitud_coseno, axis=1)

print("SIMILITUD COSENO (ProLIF) vs CRISTAL")
print("=" * 45)
print(f"  Referencia: {ref_vector.name if hasattr(ref_vector,'name') else LIGAND_CODE}")
print(f"  Similitud media: {fps_scaled['cos_sim'].mean():.3f}")
print(f"  Rango: [{fps_scaled['cos_sim'].min():.3f}, {fps_scaled['cos_sim'].max():.3f}]")
print()
print(f"  Top 5 por similitud ProLIF:")
top_prolif = fps_scaled['cos_sim'].sort_values(ascending=False).head(6)
for mol, sim in top_prolif.items():
    flag = '★ REFERENCIA' if mol == LIGAND_CODE else ''
    print(f"  {mol:<25}: {sim:.4f}  {flag}")


In [ ]:
# ── Paso 2: Normalizar el score de Vina ─────────────────────────────────────
# El score de Vina es negativo — el más negativo es el mejor
# Invertimos y normalizamos para que mayor = mejor

df_scoring = df_resultados[['molecule_chembl_id', 'std_smiles',
                              'pActividad', 'standard_type', 'vina_score']].copy()

vina_vals = df_scoring['vina_score'].values.reshape(-1, 1)

# Inversión: multiplicamos por -1 (el más negativo se vuelve el más positivo)
vina_inv = -vina_vals
scaler_vina = MinMaxScaler()
df_scoring['vina_norm'] = scaler_vina.fit_transform(vina_inv).flatten()

print("SCORE DE VINA NORMALIZADO")
print("=" * 45)
print(f"  Score Vina original — rango: [{df_scoring['vina_score'].min():.2f}, {df_scoring['vina_score'].max():.2f}] kcal/mol")
print(f"  Vina normalizado    — rango: [{df_scoring['vina_norm'].min():.3f}, {df_scoring['vina_norm'].max():.3f}]")
print(f"  (1.0 = mejor score de Vina, 0.0 = peor)")


In [ ]:
# ── Paso 3: Combinar en el scoring compuesto ─────────────────────────────────
# Añadir similitud ProLIF al DataFrame de scoring

# Alinear índices
fps_cos_dict = fps_scaled['cos_sim'].to_dict()
df_scoring['prolif_sim'] = df_scoring['molecule_chembl_id'].map(fps_cos_dict)
df_scoring['prolif_sim'] = df_scoring['prolif_sim'].fillna(0.0)

# Calcular el score compuesto
df_scoring['score_final'] = (df_scoring['vina_norm'] + df_scoring['prolif_sim']) / 2

# Ordenar por score final (mayor = mejor)
df_scoring = df_scoring.sort_values('score_final', ascending=False).reset_index(drop=True)
df_scoring.index = range(1, len(df_scoring) + 1)
df_scoring.index.name = 'Ranking'

print("RANKING FINAL — SCORING COMPUESTO")
print(f"Target: {TARGET_NAME}")
print("=" * 80)
print(f"{'#':>4}  {'ChEMBL ID':<18}  {'Vina (kcal/mol)':>15}  {'Vina norm':>9}  {'ProLIF sim':>10}  {'Score final':>11}")
print("-" * 80)
for rank, row in df_scoring.head(15).iterrows():
    print(f"  {rank:>2}.  {row['molecule_chembl_id']:<18}  "
          f"{row['vina_score']:>14.2f}  {row['vina_norm']:>9.4f}  "
          f"{row['prolif_sim']:>10.4f}  {row['score_final']:>11.4f}")


---
## 10. Análisis de resultados y selección de candidatos


In [ ]:
# ── Scatter plot: Vina score vs similitud ProLIF ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: scatter Vina vs ProLIF
ax1 = axes[0]
sc = ax1.scatter(
    df_scoring['vina_score'],
    df_scoring['prolif_sim'],
    c=df_scoring['score_final'],
    cmap='RdYlGn', s=30, alpha=0.7, edgecolors='none'
)
plt.colorbar(sc, ax=ax1, label='Score compuesto')

# Marcar el top 10
top10 = df_scoring.head(10)
ax1.scatter(top10['vina_score'], top10['prolif_sim'],
            s=80, marker='*', c='gold', edgecolors='black',
            linewidth=0.5, zorder=5, label='Top 10')
for _, row in top10.head(5).iterrows():
    ax1.annotate(row['molecule_chembl_id'][-6:],
                 (row['vina_score'], row['prolif_sim']),
                 fontsize=7, xytext=(3, 3), textcoords='offset points')

ax1.set_xlabel('Score Vina (kcal/mol)', fontsize=11)
ax1.set_ylabel('Similitud coseno ProLIF', fontsize=11)
ax1.set_title('Vina vs similitud ProLIF\n(estrella = Top 10 score compuesto)', fontsize=10)
ax1.legend(fontsize=9)
ax1.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

# Panel derecho: distribución del score compuesto
ax2 = axes[1]
ax2.hist(df_scoring['score_final'], bins=30,
         color='#38bdf8', alpha=0.8, edgecolor='white', linewidth=0.4)
ax2.axvline(df_scoring['score_final'].quantile(0.9),
            color='gold', linestyle='--', linewidth=2,
            label=f'Percentil 90 ({df_scoring["score_final"].quantile(0.9):.3f})')
ax2.set_xlabel('Score compuesto', fontsize=11)
ax2.set_ylabel('Frecuencia', fontsize=11)
ax2.set_title('Distribución del score compuesto', fontsize=10)
ax2.legend(fontsize=9)

plt.suptitle(f'Análisis de virtual screening — {TARGET_NAME}\n'
             f'({len(df_scoring)} moléculas)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{docks_directory}/analisis_scoring.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Comparar ranking por Vina solo vs score compuesto ───────────────────────
# Esto ilustra por qué el scoring compuesto cambia la selección

ranking_vina = df_scoring.sort_values('vina_score').reset_index()['molecule_chembl_id'].head(10).tolist()
ranking_comp = df_scoring.sort_values('score_final', ascending=False).reset_index()['molecule_chembl_id'].head(10).tolist()

print("COMPARACIÓN: Top 10 por Vina solo vs Score Compuesto")
print("=" * 60)
print(f"{'Pos':>3}  {'Por Vina (kcal/mol)':^25}  {'Por Score Compuesto':^25}")
print("-" * 60)
for i, (v, c) in enumerate(zip(ranking_vina, ranking_comp), 1):
    mismo = '← igual' if v == c else ''
    print(f"  {i:>2}.  {v:<25}  {c:<25} {mismo}")

moléculas_nuevas = len(set(ranking_comp) - set(ranking_vina))
print()
print(f"Moléculas en el top 10 del score compuesto")
print(f"que NO estaban en el top 10 de Vina solo: {moléculas_nuevas}")
print()
print("💡 Si este número es > 0, el scoring compuesto está añadiendo")
print("   información real sobre la especificidad de interacciones.")


In [ ]:
# ── Top candidatos: interacciones con el sitio de unión ─────────────────────
# Analizar qué residuos contacta el mejor candidato vs el cristal

print("ANÁLISIS DE INTERACCIONES — Top 3 candidatos vs cristal")
print("=" * 60)

top3 = df_scoring.head(3)['molecule_chembl_id'].tolist()
mols_analisis = [LIGAND_CODE] + top3 if LIGAND_CODE in fps_df.index else top3

for mol_id in mols_analisis:
    if mol_id not in fps_df.index:
        continue
    fp_mol = fps_df.loc[mol_id]
    interacciones = fp_mol[fp_mol > 0]
    n_int = len(interacciones)
    tag = '★ CRISTAL' if mol_id == LIGAND_CODE else f'rank #{df_scoring[df_scoring.molecule_chembl_id==mol_id].index[0] if mol_id in df_scoring.molecule_chembl_id.values else "?"}'
    print(f"\n  {mol_id} ({tag})")
    print(f"  Interacciones activas: {n_int}")
    for col, val in interacciones.items():
        col_str = str(col)
        print(f"    {col_str[:50]}: {val}")


---
## 11. Guardar los resultados finales


In [ ]:
# ── Preparar y guardar la tabla de resultados completa ──────────────────────
TARGET_SLUG = TARGET_NAME.lower().replace(' ', '_').replace('/', '_')[:30]

# Tabla completa sin la columna PDBQT (ocupa mucho espacio)
df_final = df_scoring.copy()
df_final = df_final.drop(columns=['pdbqt'], errors='ignore')

archivo_ranking = f'{docks_directory}/ranking_final_{TARGET_SLUG}.csv'
df_final.to_csv(archivo_ranking, index=True)

print("✅ ARCHIVOS GUARDADOS")
print("=" * 55)
for archivo in [
    archivo_ranking,
    f'{docks_directory}/fps_activos.csv',
    f'{docks_directory}/heatmap_interacciones.png',
    f'{docks_directory}/analisis_scoring.png'
]:
    if os.path.exists(archivo):
        tam = os.path.getsize(archivo) / 1024
        print(f"  {archivo:<50} ({tam:.0f} KB)")

print()
print("RESUMEN FINAL DEL VIRTUAL SCREENING")
print("=" * 55)
print(f"  Target:                  {TARGET_NAME}")
print(f"  Moléculas cribadas:      {len(df_scoring)}")
print(f"  Score Vina medio:        {df_scoring['vina_score'].mean():.2f} kcal/mol")
print(f"  Similitud ProLIF media:  {df_scoring['prolif_sim'].mean():.3f}")
print(f"  Score compuesto medio:   {df_scoring['score_final'].mean():.3f}")
print()
print("  TOP 5 CANDIDATOS (score compuesto):")
for rank, row in df_final.head(5).iterrows():
    print(f"  {rank:>2}. {row['molecule_chembl_id']:<20} "
          f"Vina={row['vina_score']:.2f}  ProLIF={row['prolif_sim']:.3f}  "
          f"Final={row['score_final']:.3f}")


---
## ✅ Resumen del notebook

| Paso | Función / herramienta | Resultado |
|------|----------------------|-----------|
| **Descarga ChEMBL** | `chembl_mols()` | Moléculas activas con filtros QSAR |
| **Curación** | `process_smiles()` | SMILES limpios, sin sales, canónicos |
| **Conformeros** | `get_conformers()` (ETKDGv3 + UFF) | PDB del mínimo energético por molécula |
| **PDBQT ligandos** | `prepare_ligands()` (Meeko) | Cargas Gasteiger + torsiones |
| **Docking batch** | `vina_docking()` | 5 poses × n moléculas → SDF |
| **Fingerprints** | `get_prolif()` (ProLIF) | Matriz residuo × tipo de interacción |
| **Heatmap** | seaborn | Visualización agrupada por tipo de interacción |
| **Score Vina norm.** | MinMaxScaler | Energía invertida y escalada a [0,1] |
| **Similitud ProLIF** | Similitud coseno | ¿Cómo de similar al ligando de referencia? |
| **Score compuesto** | (Vina + ProLIF) / 2 | Ranking integrado |

## ✅ Conexión con los demás notebooks del curso

```
NB-DATA-01/02  →  dataset curado de ChEMBL   ─┐
NB-DATA-03     →  espacio químico explorado    ├─→ NB-DOCK-02 (este notebook)
NB-ML-01       →  modelos QSAR activo/inactivo ─┘        ↓
NB-DOCK-01     →  protocolo validado (RMSD≤2Å)    Ranking de candidatos
```

Los candidatos del Top 10 pueden ahora:
1. Priorizarse para síntesis o compra
2. Cruzarse con los activos del modelo QSAR (NB-ML-01)
3. Visualizarse en 3D con NGLview para inspección manual

---
*NB-DOCK-02 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*  
*Pipeline basado en: ChEMBL, RDKit, Meeko, AutoDock Vina, ProLIF*
